In [ ]:
import os
import json
import re
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.ai.projects import AIProjectClient
from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import ResourceExistsError

In [ ]:
# Load environment variables from .env in this project's directory.
load_dotenv()

PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT")
# Accept MODEL_DEPLOYMENT_NAME as an alternative variable name too.
MODEL_NAME = os.getenv("MODEL_NAME") or os.getenv("MODEL_DEPLOYMENT_NAME")
EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME")

credential = DefaultAzureCredential()

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)

openai_client = project.get_openai_client()

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default",
)

In [ ]:
llm = ChatOpenAI(
    model=MODEL_NAME,
    base_url=str(openai_client.base_url),
    api_key=token_provider
)

In [ ]:
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME,
    base_url=str(openai_client.base_url),
    api_key=token_provider
)

In [ ]:
service = BlobServiceClient(
    account_url=os.environ["AZURE_STORAGE_ACCOUNT_URL"],
    credential=credential,
)

container_name = os.environ["AZURE_STORAGE_CONTAINER"]

container = None
try:
    container = service.create_container(name=container_name)
    print(f"Created container: {container_name}")
except ResourceExistsError:
    container = service.get_container_client(container_name)
    print(f"Container already exists: {container_name}")

In [ ]:
# Path to your separate JSON file
json_path = Path("data/mock_characters.json")

# Verify the file exists
if not json_path.is_file():
    raise FileNotFoundError(
        f"Could not find {json_path.resolve()}. "
        "Check that the file is in your project's data folder."
    )

# Choose the destination name in Azure Blob Storage
blob_name = "characters/mock_characters.json"

# Upload the file
blob = container.get_blob_client(blob_name)

In [ ]:
with json_path.open("rb") as file:
    blob.upload_blob(file, overwrite=True)

print(f"Successfully uploaded {json_path.name}")
print(f"Azure blob name: {blob_name}")

In [ ]:
blob_name = "characters/mock_characters.json"

blob = container.get_blob_client(blob_name)

json_bytes = blob.download_blob().readall()
characters = json.loads(json_bytes.decode("utf-8"))

print(f"Loaded {len(characters)} characters from Azure Blob Storage")
print(json.dumps(characters[:len(characters)], indent=2))

In [ ]:
question = "Which character would be the most funny, based on the information provided?"

context = json.dumps(characters, indent=2)

prompt = f"""
You are an assistant that answers questions about fictional movie characters.

Use the character data below to answer the question.
Base your answer only on the provided data.
If the data does not contain enough information, say so.
Do not invent facts.

Character data:
{context}

Question:
{question}
"""

response = llm.invoke(prompt)

print(response.content)